In [38]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
from tqdm import tqdm
import torch.nn.functional as F

In [25]:
# %pip install torch transformers pandas tqdm

In [22]:
import pandas as pd

df = pd.read_csv("go_emotions_dataset.csv")

# 1. Define strict "Happiness" vs "Unhappiness" categories for contrast
happy_categories = ["joy", "excitement", "amusement"]
negative_categories =["sadness", "grief", "disappointment", "anger", "annoyance"]

# 2. Isolate the pure positive and negative rows
# We use bitwise NOT (~) to ensure no confusing overlap (e.g. a comment that is both joyful AND sad)
happy_df = df[
    df[happy_categories].eq(1).any(axis=1) &
    ~df[negative_categories].eq(1).any(axis=1)
]

negative_df = df[
    df[negative_categories].eq(1).any(axis=1) &
    ~df[happy_categories].eq(1).any(axis=1)
]

print(f"Total available pure happy samples: {len(happy_df)}")
print(f"Total available pure negative samples: {len(negative_df)}")

# 3. Sample a small, balanced number of examples
# The SADI paper (Section 6.2) notes optimal results are achieved with ~150-500 pairs
NUM_PAIRS = 200

# Ensure we don't sample more than we have
num_to_sample = min(NUM_PAIRS, len(happy_df), len(negative_df))

happy_texts = happy_df["text"].sample(n=num_to_sample, random_state=42).tolist()
negative_texts = negative_df["text"].sample(n=num_to_sample, random_state=42).tolist()

# 4. Construct the Contrastive Pairs [(x_pos, x_neg), ...] for SADI Difference Extraction
contrastive_pairs = list(zip(happy_texts, negative_texts))

print(f"\nCreated {len(contrastive_pairs)} contrastive pairs for the SADI Happiness Mask.")
print("-" * 50)

# Display a few examples of the pairs you will feed to the LLM
for i in range(5):
    print(f"Pair {i+1}:")
    print(f"  [+] Happy  (x_pos): {contrastive_pairs[i][0]}")
    print(f"  [-] Unhappy(x_neg): {contrastive_pairs[i][1]}\n")

Total available pure happy samples: 20821
Total available pure negative samples: 33231

Created 200 contrastive pairs for the SADI Happiness Mask.
--------------------------------------------------
Pair 1:
  [+] Happy  (x_pos): Wow you found the answer, wish you were on top, will link to you in my post
  [-] Unhappy(x_neg): What the hell

Pair 2:
  [+] Happy  (x_pos): My barber was thinning my Hair with a new blade and cut my ear. Bled like crazy.
  [-] Unhappy(x_neg): [NAME] dirty talk really just Doesnt Work for me. Sounds silly at the best of times.

Pair 3:
  [+] Happy  (x_pos): *Hey just noticed..* it's your **1st Cakeday** SirVortivask! ^(hug)
  [-] Unhappy(x_neg): The thin side and endlines are really throwing me off.

Pair 4:
  [+] Happy  (x_pos): Ahaha it's his mother's fault!
  [-] Unhappy(x_neg): Ok but what if you hire him into a management position. He makes women feel uncomfortable and a discrimination suit waiting to happen.

Pair 5:
  [+] Happy  (x_pos): Literally anyon

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3-4B")
model = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen3-4B",
    torch_dtype="auto",
    device_map="auto",
    output_hidden_states=True,
)
model.eval()

The following generation flags are not valid and may be ignored: ['output_hidden_states']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Qwen3ForCausalLM(
  (model): Qwen3Model(
    (embed_tokens): Embedding(151936, 2560)
    (layers): ModuleList(
      (0-35): 36 x Qwen3DecoderLayer(
        (self_attn): Qwen3Attention(
          (q_proj): Linear(in_features=2560, out_features=4096, bias=False)
          (k_proj): Linear(in_features=2560, out_features=1024, bias=False)
          (v_proj): Linear(in_features=2560, out_features=1024, bias=False)
          (o_proj): Linear(in_features=4096, out_features=2560, bias=False)
          (q_norm): Qwen3RMSNorm((128,), eps=1e-06)
          (k_norm): Qwen3RMSNorm((128,), eps=1e-06)
        )
        (mlp): Qwen3MLP(
          (gate_proj): Linear(in_features=2560, out_features=9728, bias=False)
          (up_proj): Linear(in_features=2560, out_features=9728, bias=False)
          (down_proj): Linear(in_features=9728, out_features=2560, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen3RMSNorm((2560,), eps=1e-06)
        (post_attention_layer

In [ ]:
def get_last_token_hidden_states(text, model, tokenizer):
    prompt = f"Statement: '{text}'\nEmotion:"

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model(**inputs)

    hidden_states = outputs.hidden_states[1:]
    last_token_activations =[layer[:, -1, :] for layer in hidden_states]
    concatenated_activations = torch.cat(last_token_activations, dim=-1).squeeze()

    return concatenated_activations

def build_sadi_mask(contrastive_pairs, model, tokenizer, top_k=1000):
    N = len(contrastive_pairs)

    sum_pos = None
    sum_neg = None

    print(f"Extracting activations for {N} contrastive pairs...")
    for pos_text, neg_text in tqdm(contrastive_pairs):
        A_pos = get_last_token_hidden_states(pos_text, model, tokenizer)
        A_neg = get_last_token_hidden_states(neg_text, model, tokenizer)

        if sum_pos is None:
            sum_pos = A_pos
            sum_neg = A_neg
        else:
            sum_pos += A_pos
            sum_neg += A_neg

    # Calculate the exact center of Happy and Sad
    mean_pos = sum_pos / N
    mean_neg = sum_neg / N

    # The direction pointing from Sad to Happy
    mean_difference = mean_pos - mean_neg

    #[FIX 2]: Calculate the neutral baseline (the exact midpoint)
    mean_baseline = (mean_pos + mean_neg) / 2.0

    mask = torch.zeros_like(mean_difference)

    # Use torch.abs() to find the features that change the MOST
    _, top_k_indices = torch.topk(torch.abs(mean_difference), top_k)
    mask[top_k_indices] = 1.0

    # We now return the baseline as well!
    return mean_baseline, mean_difference, mask

# ==========================================
# 3. EXECUTION & REWARD CALCULATION
# ==========================================

# K is a hyperparameter. The paper notes K is task-specific.
# 1000 is a safe starting point for hidden states, but you can sweep this value.
K = 1000
mean_baseline, mean_diff_vector, happiness_mask = build_sadi_mask(contrastive_pairs, model, tokenizer, top_k=K)

def get_happiness_reward(user_text, model, tokenizer, baseline, mean_diff_vector, mask):
    # 1. Get activations for the user's text
    A_user = get_last_token_hidden_states(user_text, model, tokenizer)

    # 2. CENTER THE VECTOR: Subtract the baseline to anchor 'neutral' at 0
    A_centered = A_user - baseline

    # 3. Apply the mask
    masked_user = A_centered * mask
    masked_direction = mean_diff_vector * mask

    # 4. Calculate Cosine Similarity
    reward = F.cosine_similarity(masked_user.unsqueeze(0), masked_direction.unsqueeze(0)).item()

    return reward

Extracting activations for 200 contrastive pairs...


100%|██████████| 200/200 [00:08<00:00, 24.54it/s]


In [57]:
# --- Test the Reward Function ---
test_texts =[
    "I just got promoted at work and I'm absolutely thrilled!",
    "I had a normal sandwich for lunch today.",
    "My car broke down in the rain and I am furious.",
    "ok.",
    "meh.",
    "Worst day ever.",
    ":)",
    ";(",
    "rough day, but I hope tomorrow is better."
]

print("\n--- Testing the Happiness Reward Function ---")
for text in test_texts:
    score = get_happiness_reward(text, model, tokenizer, mean_baseline, mean_diff_vector, happiness_mask)
    print(f"Text: '{text}'\nReward Score: {score:.4f}\n")


--- Testing the Happiness Reward Function ---
Text: 'I just got promoted at work and I'm absolutely thrilled!'
Reward Score: 0.5508

Text: 'I had a normal sandwich for lunch today.'
Reward Score: -0.0386

Text: 'My car broke down in the rain and I am furious.'
Reward Score: -0.6992

Text: 'ok.'
Reward Score: 0.2871

Text: 'meh.'
Reward Score: -0.3340

Text: 'Worst day ever.'
Reward Score: -0.7188

Text: ':)'
Reward Score: 0.7422

Text: ';('
Reward Score: -0.4980

Text: 'rough day, but I hope tomorrow is better.'
Reward Score: -0.3867

